# QCOL Quantum App Studio — Backend (Colab)

Runs the exact same `qcol/api.py` FastAPI service used in the HuggingFace deployment, exposed with a public tunnel URL you can paste into the QCOL Founder panel → App Studio Connector.

**Before you start:** upload `qcol-app-studio-backend.zip` itself (the whole zip, not its contents) into this Colab session — click the folder icon on the left sidebar → upload icon → select the zip. The next cell unzips it automatically.

**Know the limits:** this session disconnects after ~90 minutes idle or ~12 hours max, and the public URL changes every time you restart. Good for testing and demoing to yourself before founder approval — not a permanent home for the service.

In [ ]:
# Step 0 — unzip the uploaded backend package
import zipfile, os

zip_path = "/content/qcol-app-studio-backend.zip"
assert os.path.exists(zip_path), (
    f"Could not find {zip_path} — upload the zip via the folder icon on the left "
    "sidebar first, then re-run this cell."
)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

print("Unzipped. Contents of /content:")
print(os.listdir("/content"))

In [ ]:
# Step 1 — install the exact pinned dependencies (matches the HF Space build)
!pip install -q numpy==1.26.4 scipy==1.13.1 sympy==1.13.3 matplotlib==3.8.4 \
    pandas==2.2.2 networkx==3.3 h5py==3.11.0 requests==2.32.4 \
    cirq-core==1.4.1 openfermion==1.6.1 pyqasm==1.0.4 openqasm3==1.0.1 ply==3.11 \
    qiskit==1.2.4 qiskit-aer==0.14.2 \
    fastapi==0.104.1 uvicorn==0.24.0 httpx==0.25.2 huggingface-hub==0.19.4 \
    altair==5.2.0 pillow==10.4.0 websockets==11.0.3 pydantic==2.5.3 \
    starlette==0.27.0 anyio==3.7.1 pyngrok

print('Installed. If this cell installed a different numpy/pydantic than what was already')
print('loaded, use Runtime -> Restart session once, then re-run from this cell.')

In [ ]:
# Step 2 — confirm the API imports cleanly before exposing it publicly
import sys
sys.path.insert(0, '/content')
from qcol.api import app
print('Import OK. Routes registered:', len(app.routes))

In [ ]:
# Step 3 — set your ngrok authtoken
# Free account: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below (or use Colab's Secrets panel on the left and read it from there instead).
NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [ ]:
# Step 4 — run the FastAPI server in a background thread, then open the tunnel
import threading, time, uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=7860, log_level="info")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(4)

public_url = ngrok.connect(7860, "http")
print("\nBackend is live at:", public_url)
print("\nPaste this into QCOL -> Founder panel -> App Studio Connector -> URL DEL BACKEND API")
print("Then click PROBAR to confirm the site can reach it.")

In [ ]:
# Step 5 (optional) — quick self-test from inside Colab, before testing from the site
import requests
r = requests.get(f"{public_url}/openapi.json", timeout=10)
print("Status:", r.status_code)
print(r.json().get("info", {}).get("title"))

### Keeping it alive
Leave this tab open and the runtime connected while you test. If you disconnect and reconnect, re-run cells 4-5 — you'll get a **new** public URL each time, so you'll need to update it in the Founder panel again.

### When you're ready to make it permanent
Deploy the same `qcol/` + `requirements.txt` + `Dockerfile` as a HuggingFace Space instead (see `qcol-app-studio-backend.zip`) so the URL stays stable and the service doesn't go down when you close this tab. Nothing else about the connector changes — just swap the URL in the same founder panel field.